# Network Simulator Explanation Notebook

This notebook explains the **flow of code**, **logic of each file**, and **how every scenario runs** for your Java Maven project:

- **Project:** ITL351 Computer Networks Lab - Semester Project
- **Submission 1:** Physical Layer + Data Link Layer
- **Language:** Java 22
- **Build:** Maven
- **Entry Point:** `com.ansh.networksim.Main`

> Important note: this notebook is based on the specification you provided, so the explanations reflect the intended design and behavior of the codebase described in your `.md` file.

## 1. Big Picture

The project is divided into four main parts:

| Part | Package / Files | Responsibility |
|---|---|---|
| Entry layer | `Main.java` | Takes user input, chooses scenario, builds demo/custom topology, runs protocol flow |
| Topology model | `model/*`, `network/Network.java` | Devices, links, registration, validation, and reporting |
| Data link protocols | `datalink/*` | Frames, checksum, CSMA/CD, Go-Back-N, collision handling, retransmission |
| Visualization helpers | `simulation/*` | Converts payloads to bit streams and prints per-tick traces |

### Overall control flow
![Overall Flow](overall_flow.png)

### Package relationship map
![Package Relationship Map](architecture_map.png)

## 2. How the Project Starts

### File: `src/main/java/com/ansh/networksim/Main.java`

This is the **starting point** of the whole simulator.

### What `Main` is responsible for

| Step | What happens |
|---|---|
| 1 | Read scenario name from command-line args or prompt user interactively |
| 2 | Decide whether to run a demo scenario or custom scenario |
| 3 | Ask for topology and protocol-specific input |
| 4 | Build devices and connect them using `Network` |
| 5 | Trigger physical send, frame send, CSMA/CD round, or Go-Back-N transfer |
| 6 | Print results, traces, and summaries |

### Mental model
Think of `Main` as the **director**.  
It does not implement switching, checksum, or retransmission itself.  
Instead, it:
- collects user choices,
- creates the right objects,
- calls the right methods,
- and lets each class do its own job.

### Typical code flow in `Main`
1. User selects a scenario.
2. `Main` creates a `Network`.
3. `Main` creates devices like `EndDevice`, `Hub`, `Switch`, or `Bridge`.
4. `Main` uses `Network.connect(...)` to create `Connection`s.
5. `Main` asks sender devices to transmit data.
6. Lower-level classes handle forwarding, collision control, ACKs, and retransmissions.

## 3. Core Topology Model

These files define the **objects present in the network**.

### Summary Table

| File | Main responsibility | Key idea |
|---|---|---|
| `Device.java` | Base class | Shared properties and abstract receive methods |
| `DataPacket.java` | Physical packet | Simple source-destination-payload unit |
| `Connection.java` | Link between 2 devices | Moves packets and frames |
| `EndDevice.java` | Host node | Sends/receives packets and frames |
| `Hub.java` | Physical repeater | Broadcasts to all other ports |
| `Switch.java` | Layer-2 switch | Learns MAC and forwards intelligently |
| `Bridge.java` | 2-port switch | Same logic as switch but only two ports |
| `Network.java` | Topology manager | Registers devices, validates, connects, reports |

### Design idea
The model package represents the **hardware and links** in the simulator.  
Everything else sits on top of these classes.

## 4. File-by-File Logic Explanation

### 4.1 `Device.java`

`Device` is the abstract base class for all devices.

#### Stores
- `id`
- `name`
- list of `Connection`s

#### Why it exists
Instead of rewriting common fields in `EndDevice`, `Hub`, `Switch`, and `Bridge`, they all inherit from `Device`.

#### Likely responsibilities
- hold identity,
- hold attached links,
- provide methods for connection registration,
- declare:
  - `receive(DataPacket, Connection)`
  - `receiveFrame(Frame, Connection)`

#### Core logic
Because different devices react differently to incoming data:
- a **Hub** rebroadcasts,
- a **Switch** learns and forwards,
- an **EndDevice** accepts only if destination matches,

the base class only defines the interface and leaves actual behavior to subclasses.

---

### 4.2 `DataPacket.java`

`DataPacket` is the **physical-layer data unit**.

#### Fields
| Field | Meaning |
|---|---|
| `source` | sender device name |
| `destination` | receiver device name |
| `payload` | actual message |

#### Why this is separate from `Frame`
At physical layer, you only need to model basic delivery.  
There is no MAC learning, checksum, ACK type, or sequence number here.

---

### 4.3 `Connection.java`

A `Connection` represents a **point-to-point wire** between exactly two devices.

#### Main behavior
| Method | Purpose |
|---|---|
| `transmit(...)` | send a physical-layer packet |
| `transmitFrame(...)` | send a data-link-layer frame |

#### Core logic
A connection knows both endpoints:
- `device1`
- `device2`

When one endpoint transmits:
- the connection identifies the **other endpoint**,
- then calls that device’s receive method.

#### Why this is important
This keeps link behavior centralized.  
Devices do not need to know how to "travel through the wire"; they just ask the connection to transmit.

---

### 4.4 `EndDevice.java`

This class represents a host like `D1`, `D2`, `S1`, etc.

#### Main jobs
| Action | Meaning |
|---|---|
| `send(...)` | create a `DataPacket` and send physically |
| `sendFrame(...)` | create a `Frame` and send at data link layer |
| `receive(...)` | accept only packets addressed to itself |
| `receiveFrame(...)` | accept/validate DATA or ACK frames |

#### Physical-layer logic
1. Build `DataPacket`.
2. Choose the attached connection.
3. Ask `Connection` to transmit.
4. On receive, check `destination == myName`.
5. If yes, print that message was received.

#### Data-link logic
1. Build a `Frame`.
2. Put source MAC = sender name.
3. Put destination MAC = receiver name.
4. Add sequence number, payload, checksum, frame type.
5. Send over connection.
6. On receive:
   - if ACK, print ACK reception
   - if DATA and checksum valid, accept
   - if checksum fails, report error

#### Why ACK handling is separate
ACK frames are control feedback, not user payload.  
So the host should log them separately from ordinary data.

---

### 4.5 `Hub.java`

`Hub` acts like a **physical repeater**.

#### Core rule
A hub does **not** inspect destination intelligently.  
It simply repeats what it receives on **every other port**.

#### Logic
1. Packet/frame arrives from one connection.
2. For every other connection:
   - transmit the same packet/frame outward.

#### Meaning in networking terms
A hub creates:
- one shared medium,
- one collision domain,
- one broadcast-like behavior at physical forwarding level.

#### Why this is useful in the simulator
It demonstrates how old shared-media networks behaved:
- no address learning,
- no selective forwarding,
- everyone sees the transmission.

---

### 4.6 `Switch.java`

A switch is the main intelligent forwarding device.

#### Main jobs
| Function | Meaning |
|---|---|
| MAC learning | remember source MAC and incoming port |
| known unicast forwarding | send only to correct port |
| flooding | send to all except incoming port if destination unknown |

#### MAC learning logic
Whenever a frame arrives:
- read `source MAC`,
- map it to the incoming `Connection`.

That means the switch slowly learns where each device is located.

#### Forwarding logic
- If destination MAC exists in MAC table:
  - forward only to that port.
- Otherwise:
  - flood on all other ports.

#### Why this works
The switch learns from traffic instead of needing manual configuration.

#### Algorithm idea
```text
On incoming frame:
    macTable[source] = incomingConnection
    if destination in macTable:
        forward to mapped connection
    else:
        flood to all except incomingConnection
```

---

### 4.7 `Bridge.java`

A bridge behaves almost exactly like a switch, but it is limited to **two ports**.

#### Main differences from switch
| Device | Port count | Behavior |
|---|---:|---|
| Switch | many ports | MAC learning + selective forwarding |
| Bridge | 2 ports | same idea but two-network-segment use case |

#### Additional validation
Bridge rejects more than two connections.

#### Why it exists separately
In networking theory and lab submissions, a bridge is often taught as an early form of layer-2 segmentation device.  
This makes the architecture match the syllabus more closely.

---

### 4.8 `Network.java`

This is the topology manager.

#### Responsibilities
| Task | Why it matters |
|---|---|
| register devices | build network inventory |
| connect devices | create valid topology |
| reject invalid topology | prevent self-links, duplicates, bad registration |
| print topology | show device and connection counts |
| compute broadcast domains | analyze overall network |
| compute collision domains | show segmentation effect |

#### Validation logic
From your spec, it checks:
- invalid device registration,
- self-connection,
- duplicate links,
- lookup validation.

#### Why this is strong design
Instead of scattering validation across many classes, `Network` becomes the single place that enforces topology correctness.

## 5. Data-Link Structures

These files define how data-link communication is represented.

### Summary Table

| File | Purpose | Why it matters |
|---|---|---|
| `FrameType.java` | enum for `DATA` / `ACK` | identifies control vs payload |
| `ChecksumUtil.java` | checksum calculation | error detection |
| `Frame.java` | complete data-link unit | wraps payload with header/control info |
| `ErrorInjector.java` | corruption helper | lets you demo retransmission |
| `TransmissionRequest.java` | one sender’s request | used in shared-medium contention |

### Frame structure

| Field | Meaning |
|---|---|
| source MAC | sender identity |
| destination MAC | receiver identity |
| sequence number | order tracking for Go-Back-N |
| payload | actual content |
| checksum | error detection field |
| frame type | DATA or ACK |

### Why `Frame` is richer than `DataPacket`
At physical layer, you only care about movement.  
At data link layer, you care about:
- addressing,
- errors,
- ordering,
- acknowledgments,
- flow control.

## 6. File-by-File Data-Link Logic

### 6.1 `FrameType.java`

A small enum-like file that separates:
- `DATA`
- `ACK`

This prevents ambiguous logic in receive handlers.

---

### 6.2 `ChecksumUtil.java`

Computes a simple **modulo-256 checksum** over payload characters.

#### Algorithm
1. Convert payload into characters.
2. Sum their values.
3. Take result modulo 256.

#### Why this is enough here
This is not meant to be industrial-grade CRC.  
It is a lab-friendly checksum that is:
- easy to implement,
- easy to explain,
- enough to demonstrate error detection.

#### Conceptual formula
```text
checksum = (sum of payload byte/char values) mod 256
```

#### Receive-side logic
The receiver recomputes checksum and compares it with the frame’s stored checksum.
- If equal → accept
- If not equal → detect error

---

### 6.3 `Frame.java`

This is the main transport object at layer 2.

#### Likely methods
| Method | Purpose |
|---|---|
| create DATA frame | payload + seq + checksum |
| create ACK frame | acknowledgment control frame |
| validate checksum | receiver-side integrity check |
| corrupt payload | for testing/demo |

#### Important design role
This class centralizes frame creation and validation, which keeps logic clean in `EndDevice`, `Switch`, and `GoBackNProtocol`.

---

### 6.4 `ErrorInjector.java`

This class intentionally corrupts selected sequence numbers **once**.

#### Why it exists
To demonstrate:
- checksum failure,
- duplicate ACK behavior,
- retransmission,
- Go-Back-N recovery.

Without controlled corruption, the protocol demo would always succeed too easily.

#### Typical behavior
- choose a target sequence number,
- corrupt it when first encountered,
- remember that it has already been corrupted,
- do not corrupt the same sequence repeatedly unless designed to.

---

### 6.5 `TransmissionRequest.java`

Represents one sender attempting to access a shared medium.

#### Likely fields
| Field | Meaning |
|---|---|
| sender | who wants to transmit |
| frame | what is being transmitted |
| retry count | how many attempts already used |
| maybe backoff slot | when it will retry |

#### Why needed
Instead of directly letting devices fight for the medium, the system models contention as a list of requests.  
That makes collision simulation structured and testable.

## 7. CSMA/CD: Logic and Flow

### Files involved
- `SharedMedium.java`
- `CsmaCdAccessControl.java`
- `TransmissionRequest.java`
- `TransmissionTraceRenderer.java`

### Scenario flow diagram
![CSMA/CD Flow](csma_flow.png)

### 7.1 `SharedMedium.java`

This class models a **shared wire** where multiple senders may try to transmit in the same time slot.

#### Stores
| State | Purpose |
|---|---|
| medium name | identification |
| current tick | simulation time |
| busy/idle flag | whether medium is occupied |
| active requests | current contenders |

#### Main idea
It is the shared environment that all competing senders observe.

---

### 7.2 `CsmaCdAccessControl.java`

This class simulates **Carrier Sense Multiple Access with Collision Detection**.

### Real-world idea
In a shared medium:
1. devices sense the channel,
2. if free, they send,
3. if two send together, collision occurs,
4. they stop and retry later.

### Simulator algorithm
From your spec:

1. Start a transmission round on `SharedMedium`.
2. If one sender is active, it transmits successfully.
3. If multiple senders are active, collision is declared.
4. Jam signal is shown.
5. Binary exponential backoff slot is selected for each sender.
6. Earliest contenders retry first.
7. If multiple retry together, they collide again.

### Logic breakdown

| Step | Internal meaning |
|---|---|
| Collision detection | count active senders in current slot |
| Jam signal | explicit log that collision was seen by all |
| Backoff | each sender waits a selected number of rounds |
| Retry tracking | each request increments retry count |
| Bounded retries | fail permanently if max retry is exceeded |

### Why deterministic backoff is useful
In real Ethernet, backoff is randomized.  
In your simulator, deterministic backoff makes the demo:
- repeatable,
- testable,
- easier to verify in JUnit.

### Pseudocode
```text
collect requests for current tick
if activeRequests == 1:
    transmit successfully
else:
    log collision
    emit jam signal
    for each request:
        retryCount++
        choose backoff slot
        if retryCount > limit:
            mark failed
schedule next retry rounds
```

## 8. Go-Back-N: Logic and Flow

### Files involved
- `GoBackNProtocol.java`
- `Frame.java`
- `ChecksumUtil.java`
- `ErrorInjector.java`
- `TransmissionTraceRenderer.java`

### Scenario flow diagram
![Go-Back-N Flow](gobackn_flow.png)

### What problem Go-Back-N solves
It is a **sliding-window flow control and error recovery protocol**.

Instead of sending one frame and waiting every time, the sender can keep several frames in flight.

### Sender-side state
Typical variables in Go-Back-N:

| Variable | Meaning |
|---|---|
| `base` | first unacknowledged frame |
| `nextSeqNum` | next sequence number available to send |
| `windowSize` | max outstanding unacknowledged frames |
| timer(s) | used to detect timeout |

### Receiver-side rule
The receiver accepts **only the next expected sequence number**.

That means:
- if expected frame arrives correctly → accept and ACK cumulatively
- if a later frame arrives first → reject as out-of-order
- if corrupted frame arrives → reject and do not advance expected sequence

### High-level algorithm

1. Send all frames in current window.
2. Start timers.
3. Receiver validates each frame.
4. Good in-order frames get accepted.
5. Receiver sends cumulative ACKs.
6. If one frame is corrupted or missing, later frames are not fully accepted.
7. Sender times out on the oldest missing frame.
8. Sender retransmits from `base`.

### Why it is called Go-Back-N
If frame `k` is lost or corrupted, sender goes back to `k` and retransmits that frame and all later unacknowledged frames.

### Pseudocode
```text
while base < totalFrames:
    while nextSeqNum < base + windowSize and nextSeqNum < totalFrames:
        send frame[nextSeqNum]
        start timer if needed
        nextSeqNum++

    wait for ACK or timeout

    if cumulative ACK received for seq = a:
        base = a + 1
    else if timeout at base:
        retransmit frames from base to nextSeqNum - 1
```

### Why checksum and error injector matter here
Go-Back-N only becomes meaningful when:
- corruption can happen,
- receiver can detect it,
- sender reacts to missing ACK progress.

That is why `ChecksumUtil` and `ErrorInjector` are central to this scenario.

## 9. Visualization Helpers

### Files
- `PayloadUtil.java`
- `BitStream.java`
- `FrameSerializer.java`
- `TransmissionTraceRenderer.java`

These are not forwarding devices or protocols.  
They are **explanation and trace tools**.

### 9.1 `PayloadUtil.java`
Converts user input into bit form.

| Input type | Output |
|---|---|
| message | each character becomes 8-bit binary |
| raw bits | preserved as-is |

This is useful because your simulator can show what travels bit by bit.

### 9.2 `BitStream.java`
A wrapper around a bit sequence.

Why use a wrapper instead of plain string?
- cleaner abstraction,
- easier future extensions,
- easier renderer integration.

### 9.3 `FrameSerializer.java`
Converts a frame into a serialized bit stream.

This supports:
- future extensibility,
- full-frame visualization,
- consistent representation.

Even if current visible traces emphasize payload bits, this file gives you a path toward full frame-level visualization later.

### 9.4 `TransmissionTraceRenderer.java`

This is the file that makes the simulator **easy to understand visually**.

#### It prints:
- payload bits,
- one bit per tick,
- collision overlaps,
- jam signal bits,
- ACK frames with no payload.

### Why it is useful for viva/lab understanding
Without this renderer, logs only say "frame sent" or "collision happened."  
With it, you can actually see:
- the sequence of bits,
- which tick they occurred on,
- where collisions overlap,
- when ACK frames appear.

## 10. Scenario-by-Scenario Code Flow

This section explains **which files get involved** and **how control moves** in each scenario.

---

## Scenario 1: Two End Devices with Dedicated Connection

### Goal
Demonstrate the simplest physical-layer communication.

### Diagram
![Physical Flow](physical_flow.png)

### Main participating files
| File | Role in this scenario |
|---|---|
| `Main.java` | builds the demo |
| `Network.java` | registers and connects the two devices |
| `EndDevice.java` | sender and receiver |
| `Connection.java` | link between the two |
| `DataPacket.java` | message unit |

### Step-by-step flow
1. `Main` creates two `EndDevice`s, for example `D1` and `D2`.
2. `Network.connect(D1, D2)` creates one `Connection`.
3. `D1.send("D2", payload)` is called.
4. `EndDevice.send(...)` builds a `DataPacket`.
5. `Connection.transmit(...)` delivers it to `D2`.
6. `D2.receive(...)` checks that destination equals `D2`.
7. Message is accepted and logged.

### Key idea
No shared media logic, no MAC learning, no checksum complexity.  
This is the base communication path.

---

## Scenario 2: Star Topology with One Hub and Five End Devices

### Goal
Show physical-layer broadcast/repeater behavior.

### Main participating files
| File | Role |
|---|---|
| `Main.java` | creates hub topology |
| `Network.java` | registers hub + 5 devices |
| `Hub.java` | repeats incoming packet/frame |
| `EndDevice.java` | sends and receives |
| `Connection.java` | each host-hub link |
| `DataPacket.java` | physical payload |

### Step-by-step flow
1. `Main` creates `H1` and five end devices.
2. Each device is connected to `H1`.
3. One sender calls `send(...)`.
4. Packet reaches the hub through one connection.
5. `Hub.receive(...)` loops through all other connections.
6. Same packet is transmitted to every other device.
7. Only the intended destination logically accepts it, but all ports see the traffic pattern.

### Key learning point
A hub is not selective.  
It behaves like a repeater, not an intelligent forwarding device.

---

## Scenario 3: Switch with Five End Devices

### Goal
Show layer-2 forwarding and address learning.

### Diagram
![Switch Flow](switch_flow.png)

### Main participating files
| File | Role |
|---|---|
| `Main.java` | builds switching scenario |
| `Network.java` | topology creation |
| `EndDevice.java` | creates frames |
| `Frame.java` | DATA/ACK unit |
| `Switch.java` | MAC learning + forwarding |
| `ChecksumUtil.java` | frame validation |
| `Connection.java` | carries frames |

### Step-by-step flow
1. `Main` creates switch `SW1` and five end devices.
2. Devices are connected to the switch.
3. Sender creates a `Frame`.
4. Frame reaches `Switch.receiveFrame(...)`.
5. Switch stores `source MAC -> incoming connection`.
6. Switch checks whether destination MAC is already known.
7. If known:
   - forward only to that port.
8. If unknown:
   - flood to all ports except the incoming one.
9. Receiver validates checksum and processes the frame.

### Key learning point
The switch becomes smarter over time.  
Initial unknown traffic may flood, but later traffic becomes selective.

---

## Scenario 4: Two Simultaneous Transmitters on a Shared Medium (CSMA/CD)

### Goal
Show collision, jam signal, and backoff.

### Main participating files
| File | Role |
|---|---|
| `Main.java` | triggers collision demo |
| `TransmissionRequest.java` | models sender intent |
| `SharedMedium.java` | shared channel state |
| `CsmaCdAccessControl.java` | collision handling |
| `TransmissionTraceRenderer.java` | tick-by-tick trace |

### Step-by-step flow
1. `Main` prepares two transmitters in same slot.
2. Each becomes a `TransmissionRequest`.
3. `CsmaCdAccessControl` starts a round on `SharedMedium`.
4. Multiple active requests are detected.
5. Collision is declared.
6. Renderer shows overlapping bits.
7. Jam signal is logged.
8. Retry counts increase.
9. Backoff slots are assigned.
10. Requests retry later.
11. Eventually one succeeds, or a request fails after max retries.

### Key learning point
This scenario demonstrates why shared media are inefficient under contention.

---

## Scenario 5: Go-Back-N Transfer with Injected Error

### Goal
Show sliding window, checksum error detection, cumulative ACKs, and retransmission.

### Main participating files
| File | Role |
|---|---|
| `Main.java` | reads window size and error injection choice |
| `PayloadUtil.java` | converts payload to bits |
| `GoBackNProtocol.java` | sender/receiver flow |
| `Frame.java` | sequence-numbered frames |
| `ChecksumUtil.java` | detect corruption |
| `ErrorInjector.java` | intentionally corrupt a chosen sequence |
| `TransmissionTraceRenderer.java` | bit trace |
| `EndDevice.java` | sender/receiver endpoints |

### Step-by-step flow
1. Payload is converted into bit stream.
2. Sender segments data into frames with sequence numbers.
3. Sender opens the first window.
4. Frames in the window are transmitted.
5. Receiver checks each frame:
   - if checksum valid and in order → accept
   - if corrupted → reject
   - if out-of-order → reject or repeat previous ACK
6. Receiver sends cumulative ACKs for the highest in-order sequence.
7. Sender advances base if ACKs move forward.
8. If a chosen frame was corrupted, ACK progress stalls.
9. Timeout occurs at the sender’s base.
10. Sender retransmits from that sequence onward.
11. Transfer completes with final statistics.

### Key learning point
Go-Back-N trades efficiency for simplicity:
- receiver keeps only the next expected frame,
- sender may resend multiple frames after one error.

---

## Scenario 6: Two Hub-Based Stars Connected by a Switch

### Goal
Combine physical broadcast segments with a data-link segmentation device.

### Diagram
![Two Star Topologies via Switch](two_stars_flow.png)

### Main participating files
| File | Role |
|---|---|
| `Main.java` | builds both star segments and switch |
| `Hub.java` | repeats inside each star |
| `Switch.java` | learns which hub-side leads to which hosts |
| `EndDevice.java` | endpoints |
| `Connection.java` | links |
| `Frame.java` | layer-2 data unit |

### Step-by-step flow
1. `Main` creates two hubs.
2. Several end devices connect to each hub.
3. Each hub has an uplink to the switch.
4. Sender in one star sends a frame.
5. Local hub repeats to all local ports and uplink.
6. Switch learns the source on that uplink side.
7. If destination on other side is not known yet, switch floods.
8. After learning, switch forwards only toward the correct hub.
9. Destination hub repeats inside its local star.
10. Receiver accepts the frame.

### Key learning point
This scenario shows how a switch reduces unnecessary traffic between segments, even if hubs still repeat traffic inside each local segment.

## 11. Domain Reporting Logic

Your project also computes broadcast and collision domains.

### Broadcast domains
From your spec:
- computed as connected components,
- because there are no layer-3 devices.

### Why that makes sense
Without routers, the whole connected layer-2 structure generally belongs to one broadcast domain, unless logic explicitly separates it.

### Collision domains
Computed by treating:
- switches and bridges as collision boundaries.

### Networking meaning
| Device | Collision domain effect |
|---|---|
| Hub | keeps shared collision behavior |
| Switch | separates collision domains per port |
| Bridge | separates collision domains across its 2 ports |

### Why this is useful in viva
It shows that your project is not just simulating packet movement; it also models the **network-design consequences** of different devices.

## 12. Automated Tests: What They Prove

Your tests are strong because they map to expected network behaviors.

### Coverage summary

| Test class | What it proves |
|---|---|
| `NetworkTest` | topology validation and safe network construction |
| `ConnectionTest` | endpoint resolution and invalid endpoint handling |
| `EndDeviceTest` | sending/receiving rules and ACK handling |
| `HubTest` | hub rebroadcast behavior |
| `SwitchTest` | MAC table learning |
| `BridgeTest` | 2-port restriction and learning |
| `SharedMediumTest` | collision detection and medium state |
| `CsmaCdAccessControlTest` | collision + backoff logic correctness |
| `GoBackNProtocolTest` | retransmission after injected error |
| `MainTest` | scenario runner and interactive flow |

### Why this matters
A good lab project is not only about writing classes.  
It is about proving that each behavior works in isolation and in integrated scenarios.

## 13. How to Understand the Code Fast During Exam or Viva

Use this reading order:

| Order | File(s) | Why first |
|---:|---|---|
| 1 | `Main.java` | shows all scenarios and entry flow |
| 2 | `Network.java` | explains topology creation and validation |
| 3 | `Device.java`, `Connection.java`, `EndDevice.java` | basic communication path |
| 4 | `Hub.java`, `Switch.java`, `Bridge.java` | forwarding device behavior |
| 5 | `Frame.java`, `ChecksumUtil.java`, `FrameType.java` | data-link internals |
| 6 | `SharedMedium.java`, `CsmaCdAccessControl.java` | access control flow |
| 7 | `GoBackNProtocol.java`, `ErrorInjector.java` | flow control and retransmission |
| 8 | `simulation/*` | visualization and logs |
| 9 | `test/*` | confirms expected behavior |

### Quick memory trick
- `Main` = director  
- `Network` = builder + validator  
- `Connection` = wire  
- `EndDevice` = host  
- `Hub` = repeat everywhere  
- `Switch/Bridge` = learn and forward  
- `Frame` = structured L2 message  
- `Checksum` = detect errors  
- `CSMA/CD` = manage collisions  
- `Go-Back-N` = recover from errors with sliding window  
- `Renderer` = show bits and timing

## 14. End-to-End Understanding in One Table

| Scenario | Main unit sent | Core forwarding device | Main algorithm shown | Main learning outcome |
|---|---|---|---|---|
| Two end devices | `DataPacket` | none | direct physical delivery | simplest send/receive |
| Hub star | `DataPacket` | `Hub` | rebroadcast | shared medium / non-intelligent repeating |
| Switch star | `Frame` | `Switch` | MAC learning + flooding | intelligent L2 forwarding |
| CSMA/CD | `Frame` / request | shared medium | collision + backoff | access control on shared medium |
| Go-Back-N | `Frame` | endpoints + protocol | sliding window retransmission | flow control + error recovery |
| Two hub stars + switch | `Frame` | `Hub` + `Switch` | local repeat + segmented forwarding | mixed topology behavior |

## 15. Final Concept Map

The simulator can be understood as a layered teaching tool:

1. **Physical Layer**
   - Data moves as `DataPacket`
   - Hubs repeat traffic
   - Connections carry data

2. **Data Link Layer**
   - Data moves as `Frame`
   - Switches/bridges learn MAC addresses
   - Checksum detects errors
   - CSMA/CD controls shared-medium access
   - Go-Back-N handles reliable windowed transfer

3. **Visualization + Testing**
   - Bit-level traces make the flow visible
   - Automated tests prove the logic works

### Final takeaway
Your project is not just a set of classes.  
It is a clean simulation pipeline:

**User choice → topology creation → packet/frame generation → forwarding behavior → error/access/flow control → trace + summary**